# Full Ensemble Training Pipeline
## Complete PPO + GRU + LightGBM Training

This notebook executes the complete ensemble training pipeline with full PPO implementation.

**Requirements:**
- Run `pip install -r requirements-training.txt` before starting
- Ensure training data is available in `data/` directory
- AWS credentials configured for S3 export (optional)

**Just run all cells to train the complete ensemble!**

In [1]:
# Environment Setup and Validation
import os
import sys
from pathlib import Path
from typing import Dict, List
import warnings
import subprocess
import importlib
import importlib.util

from packaging.version import InvalidVersion, Version

warnings.filterwarnings("ignore")

print("🔧 Setting up training environment...")

project_root = Path("/notebooks/bot") if Path("/notebooks/bot").exists() else Path.cwd()
os.chdir(project_root)
sys.path.insert(0, str(project_root))

print(f"📁 Working directory: {project_root}")

DEPENDENCIES: Dict[str, Dict[str, str]] = {
    "numpy": {"module": "numpy", "target": "1.26.4", "install": "numpy==1.26.4"},
    "scipy": {"module": "scipy", "target": "1.13.1", "install": "scipy==1.13.1"},
    "pandas": {"module": "pandas", "target": "2.2.3", "install": "pandas==2.2.3"},
    "scikit_learn": {"module": "sklearn", "target": "1.5.2", "install": "scikit-learn==1.5.2", "constraint": "scikit-learn"},
    "torch": {"module": "torch", "target": "2.3.1", "install": "torch==2.3.1"},
    "torchvision": {"module": "torchvision", "target": "0.18.1", "install": "torchvision==0.18.1"},
    "torchaudio": {"module": "torchaudio", "target": "2.3.1", "install": "torchaudio==2.3.1"},
    "lightgbm": {"module": "lightgbm", "target": "4.4.0", "install": "lightgbm==4.4.0"},
    "stable_baselines3": {"module": "stable_baselines3", "target": "2.7.0", "install": "stable-baselines3[extra]==2.7.0", "constraint": "stable-baselines3"},
    "gymnasium": {"module": "gymnasium", "target": "1.0.0", "install": "gymnasium==1.0.0"},
    "optuna": {"module": "optuna", "target": "3.6.2", "install": "optuna==3.6.2"},
    "ta": {"module": "ta", "target": "0.11.0", "install": "ta==0.11.0"},
    "matplotlib": {"module": "matplotlib", "target": "3.8.4", "install": "matplotlib==3.8.4"},
    "protobuf": {"module": "google.protobuf", "target": "4.25.3", "install": "protobuf==4.25.3", "constraint": "protobuf"},
    "tensorboard": {"module": "tensorboard", "target": "2.15.2", "install": "tensorboard==2.15.2"},
    "fsspec": {"module": "fsspec", "target": "2025.9.0", "install": "fsspec==2025.9.0"},
}

INSTALL_ORDER: List[str] = [
    "numpy",
    "scipy",
    "pandas",
    "scikit_learn",
    "torch",
    "torchvision",
    "torchaudio",
    "lightgbm",
    "stable_baselines3",
    "gymnasium",
    "optuna",
    "ta",
    "matplotlib",
    "protobuf",
    "tensorboard",
    "fsspec",
]

CLEANUP_PACKAGES: List[str] = [
    "gradient",
    "tensorflow",
    "torchvision",
    "torchaudio",
    "torch",
    "lightgbm",
    "scikit-learn",
    "scipy",
    "pandas",
    "numpy",
    "stable-baselines3",
    "gymnasium",
    "protobuf",
    "tensorboard",
    "fsspec",
]

CONSTRAINT_PATH = project_root / ".pinned_constraints.txt"

os.environ.setdefault("PIP_ROOT_USER_ACTION", "ignore")
os.environ.setdefault("PYTHONWARNINGS", "ignore")

def compare_versions(installed: str, target: str) -> bool:
    try:
        return Version(installed) == Version(target)
    except InvalidVersion:
        return False

def detect_dependency_gaps() -> Dict[str, List[str]]:
    missing: List[str] = []
    mismatched: List[str] = []
    for key, cfg in DEPENDENCIES.items():
        module_name = cfg["module"]
        spec = importlib.util.find_spec(module_name)
        if spec is None:
            missing.append(key)
            continue
        try:
            module = importlib.import_module(module_name)
            version_str = getattr(module, "__version__", None)
        except Exception:
            version_str = None
        if version_str is None:
            mismatched.append(key)
            continue
        if not compare_versions(version_str, cfg["target"]):
            mismatched.append(key)
    return {"missing": missing, "mismatched": mismatched}

def run_pip(args: List[str], check: bool = True) -> None:
    cmd = [sys.executable, "-m", "pip"] + args
    print(f"   -> {' '.join(args)}")
    subprocess.run(cmd, check=check)

for pkg in CLEANUP_PACKAGES:
    run_pip(["uninstall", "-y", pkg], check=False)

status = detect_dependency_gaps()
needs_install = list(dict.fromkeys([*status["missing"], *status["mismatched"]]))

if needs_install:
    modules = [DEPENDENCIES[name]["module"] for name in needs_install]
    print(f"❌ Missing/outdated dependencies detected: {', '.join(modules)}")
    print("📦 Installing/upgrading required packages...")
    for key in INSTALL_ORDER:
        spec = DEPENDENCIES[key]["install"]
        run_pip(["install", "--upgrade", "--no-cache-dir", spec])

    constraint_lines = []
    for key, cfg in DEPENDENCIES.items():
        pkg_name = cfg.get("constraint") or cfg["module"].split('.')[-1]
        constraint_lines.append(f"{pkg_name}=={cfg['target']}")
    CONSTRAINT_PATH.write_text("\n".join(constraint_lines) + "\n")

    for req_file in ["requirements.txt", "requirements-training.txt", "paperspace_mlops/requirements_paperspace.txt"]:
        if Path(req_file).exists():
            run_pip([
                "install",
                "--upgrade",
                "--no-cache-dir",
                "--constraint",
                str(CONSTRAINT_PATH),
                "-r",
                req_file,
            ])

status = detect_dependency_gaps()
remaining = list(dict.fromkeys([*status["missing"], *status["mismatched"]]))
if remaining:
    unresolved = ", ".join(DEPENDENCIES[name]["module"] for name in remaining)
    print(f"⚠️ Still unresolved dependencies: {unresolved}")
    print("⚠️ Please install missing dependencies before proceeding")
    raise RuntimeError("Missing critical dependencies - training cannot proceed")

print("✅ Dependencies ready for training")

import numpy  # noqa: F401
import pandas  # noqa: F401
import torch  # noqa: F401
import lightgbm  # noqa: F401
import stable_baselines3  # noqa: F401
import gymnasium  # noqa: F401
import optuna  # noqa: F401
import ta  # noqa: F401

# Check GPU availability
gpu_available = torch.cuda.is_available()
if gpu_available:
    print(f"🚀 GPU detected: {torch.cuda.get_device_name(0)}")
else:
    print("🖥️ Using CPU for training")

# Check AWS credentials (optional)
aws_available = all(os.getenv(key) for key in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"])
if aws_available:
    print("☁️ AWS credentials available - S3 export enabled")
else:
    print("⚠️ AWS credentials missing - S3 export will be disabled")

print("\n🎯 Environment validation complete!")



🔧 Setting up training environment...
📁 Working directory: /notebooks/bot


2025-09-24 08:37:42.425014: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-09-24 08:37:42.425085: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-09-24 08:37:42.426170: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-09-24 08:37:42.433272: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-24 08:37:43.458699: W tensorflow/compiler/tf2

✅ All core dependencies available
🚀 GPU detected: NVIDIA RTX A4000
⚠️ AWS credentials missing - S3 export will be disabled

🎯 Environment validation complete!
✅ Ready for ensemble training


In [2]:
# Database Validation - Critical for Training Success
import sqlite3
from pathlib import Path

print("\n🔍 Validating training databases...")

# Expected symbols and their database files
EXPECTED_SYMBOLS = ['BTCEUR', 'ETHEUR', 'ADAEUR', 'DOTEUR', 'LINKEUR']
data_dir = Path("data")

# Check data directory exists
if not data_dir.exists():
    raise FileNotFoundError(f"❌ Data directory not found: {data_dir}")
    
print(f"✅ Data directory found: {data_dir}")

# Validate each database file
missing_databases = []
valid_databases = []
database_info = []

for symbol in EXPECTED_SYMBOLS:
    db_file = data_dir / f"{symbol.lower()}_30m.db"
    
    if not db_file.exists():
        missing_databases.append(symbol)
        print(f"❌ Missing database: {db_file}")
    else:
        # Check database can be opened and has data
        try:
            with sqlite3.connect(str(db_file)) as conn:
                cursor = conn.cursor()
                cursor.execute("SELECT COUNT(*) FROM market_data")
                row_count = cursor.fetchone()[0]
                
                # Get date range
                cursor.execute("SELECT MIN(datetime), MAX(datetime) FROM market_data")
                date_range = cursor.fetchone()
                
                valid_databases.append(symbol)
                database_info.append(f"  {symbol}: {row_count:,} records ({date_range[0]} to {date_range[1]})")
                print(f"✅ {symbol}: {row_count:,} records")
                
        except Exception as e:
            missing_databases.append(symbol)
            print(f"❌ Database error for {symbol}: {e}")

print(f"\n📊 Database Validation Summary:")
print(f"✅ Valid databases: {len(valid_databases)}/{len(EXPECTED_SYMBOLS)}")
print(f"❌ Missing databases: {len(missing_databases)}")

if database_info:
    print(f"\n📈 Database Details:")
    for info in database_info:
        print(info)

# Stop training if databases are missing
if missing_databases:
    error_msg = f"❌ CRITICAL: Training cannot proceed - Missing databases for: {', '.join(missing_databases)}"
    print(f"\n{error_msg}")
    print("\n💡 To fix this issue:")
    print("• Databases should be created on the production server, not this training machine")
    print("• Contact the system administrator to provide the missing database files")
    print("• Expected location: data/[symbol]_30m.db (e.g., data/btceur_30m.db)")
    raise RuntimeError(error_msg)
else:
    print(f"\n✅ All required databases present - training can proceed!")
    print("🚀 Ready to start ensemble training pipeline")


🔍 Validating training databases...
✅ Data directory found: data
✅ BTCEUR: 17,520 records
✅ ETHEUR: 17,520 records
✅ ADAEUR: 17,520 records
✅ DOTEUR: 17,520 records
✅ LINKEUR: 17,520 records

📊 Database Validation Summary:
✅ Valid databases: 5/5
❌ Missing databases: 0

📈 Database Details:
  BTCEUR: 17,520 records (2024-09-15T15:30:00 to 2025-09-15T15:00:00)
  ETHEUR: 17,520 records (2024-09-15T15:30:00 to 2025-09-15T15:00:00)
  ADAEUR: 17,520 records (2024-09-15T15:30:00 to 2025-09-15T15:00:00)
  DOTEUR: 17,520 records (2024-09-15T15:30:00 to 2025-09-15T15:00:00)
  LINKEUR: 17,520 records (2024-09-15T15:30:00 to 2025-09-15T15:00:00)

✅ All required databases present - training can proceed!
🚀 Ready to start ensemble training pipeline


In [ ]:
# Execute Full Ensemble Training
import time
from datetime import datetime

print("🚀 Starting Full Ensemble Training Pipeline")
print(f"⏰ Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*60)

try:
    # Import and initialize the training runner
    from paperspace_mlops.paperspace_superior_training import PaperspaceTrainingRunner
    
    print("🔧 Initializing training runner...")
    runner = PaperspaceTrainingRunner()
    
    print("\n🎯 Launching full ensemble training...")
    print("Models: PPO + GRU + LightGBM")
    print("Symbols: BTCEUR, ETHEUR, ADAEUR, DOTEUR, LINKEUR")
    print("Features: 103 (PPO) / 100 (GRU/LightGBM)")
    print("Transaction Cost: 0.25%")
    print("")
    
    # Execute training with progress tracking
    start_time = time.time()
    
    result = runner.run_training(
        symbols=None,  # Use config defaults
        models=None,   # Use config defaults (PPO + GRU + LightGBM)
        quick_test=False  # Full training
    )
    
    training_duration = time.time() - start_time
    
    print("\n" + "="*60)
    print("🏆 TRAINING RESULTS")
    print("="*60)
    
    if result["status"] == "complete":
        print("✅ Training completed successfully!")
        print(f"⏱️ Duration: {training_duration/60:.1f} minutes")
        print(f"📊 Symbols trained: {len(result['symbols_trained'])}")
        print(f"🤖 Models trained: {len(result['models_trained'])}")
        print(f"📈 Symbols: {', '.join(result['symbols_trained'])}")
        print(f"🧠 Models: {', '.join(result['models_trained'])}")
        
        # S3 Export Status
        export_status = result.get('export_status', {})
        if export_status.get('export_enabled', False):
            print(f"☁️ S3 Export: ✅ {export_status.get('models_exported', 0)} models exported")
        else:
            print("☁️ S3 Export: ⚠️ Disabled (missing AWS credentials)")
            
    else:
        print("❌ Training failed or incomplete")
        print(f"Status: {result['status']}")
        if result.get('errors'):
            print(f"Errors: {result['errors']}")
    
    print(f"\n🎉 Training pipeline completed at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
except Exception as e:
    print(f"\n💥 Training pipeline failed: {e}")
    import traceback
    traceback.print_exc()
    
    print("\n🔍 Troubleshooting tips:")
    print("• Check that data databases exist in data/ directory")
    print("• Verify all dependencies are installed")
    print("• Check available memory and disk space")
    print("• Review logs above for specific error details")

2025-09-24 08:37:44,505 - INFO - Validating Paperspace environment...
2025-09-24 08:37:44,506 - INFO - GPU detected: NVIDIA RTX A4000 (Count: 1)
2025-09-24 08:37:44,506 - WARNING - AWS credentials missing (AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_DEFAULT_REGION) - S3 export will be disabled
2025-09-24 08:37:44,507 - INFO - Available memory: 44.1 GB
2025-09-24 08:37:44,507 - INFO - Environment validation complete
2025-09-24 08:37:44,507 - INFO - Initializing trainer with config: /notebooks/bot/config/training_config.yaml


🚀 Starting Full Ensemble Training Pipeline
⏰ Start time: 2025-09-24 08:37:44
🔧 Initializing training runner...

🎯 Launching full ensemble training...
Models: PPO + GRU + LightGBM
Symbols: BTCEUR, ETHEUR, ADAEUR, DOTEUR, LINKEUR
Features: 103 (PPO) / 100 (GRU/LightGBM)
Transaction Cost: 0.25%



2025-09-24 08:37:44,526 - INFO - 📄 Loaded training config keys: ['models', 'random_seed', 'validation_split', 'test_split', 'cv_splits', 'embargo_period', 'max_workers', 'batch_size', 'epochs', 'early_stopping_patience', 'optuna_trials', 'optuna_timeout', 'memory_limit', 'gpu_enabled', 'gpu_memory_fraction']
2025-09-24 08:37:44,527 - INFO - 📄 Loaded data config keys: ['symbols', 'interval', 'lookback_days', 'use_local_databases', 'data_sources', 'output_directory', 'cache_duration']
2025-09-24 08:37:44,527 - INFO - 🔧 Sanitized training config (pre-pop): {'cv_splits': 5, 'embargo_period': 24, 'gpu_enabled': True, 'max_workers': 8, 'memory_limit': '5GB', 'models': ['ppo', 'gru', 'lightgbm'], 'optuna_timeout': 7200, 'optuna_trials': 100, 'test_split': 0.1, 'validation_split': 0.2}
2025-09-24 08:37:44,527 - INFO - 🔧 Sanitized training config (post-pop): {'cv_splits': 5, 'embargo_period': 24, 'gpu_enabled': True, 'interval': '30m', 'lookback_days': 365, 'max_workers': 8, 'memory_limit': '5G

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


2025-09-24 08:38:20,157 - INFO - ✅ Generated 197 trading features
2025-09-24 08:38:20,180 - INFO - 🎯 Selected top 100 features from 189
2025-09-24 08:38:20,186 - INFO - ✅ Generated 100 GRU features
2025-09-24 08:38:20,187 - INFO - 🎯 Creating trading-optimized targets
2025-09-24 08:38:30,783 - INFO - ✅ Created 100 trading targets
2025-09-24 08:38:30,801 - INFO - 🚀 Training GRU model for BTCEUR
2025-09-24 08:38:30,812 - INFO - 📊 Prepared data: 17520 samples, 100 features
2025-09-24 08:38:30,812 - INFO - 📈 Data split: train=12264, val=3504, test=1752
2025-09-24 08:38:30,813 - INFO - 🎯 Optimizing GRU hyperparameters
[I 2025-09-24 08:38:30,813] A new study created in memory with name: no-name-5b545133-8e30-4baa-a55b-62edaf3fb505
[I 2025-09-24 08:38:34,104] Trial 0 finished with value: -0.011904519842920593 and parameters: {'hidden_size': 132, 'num_layers': 2, 'dropout': 0.3228570144574344, 'learning_rate': 0.005563517145765826}. Best is trial 0 with value: -0.011904519842920593.
[I 2025-09-

In [ ]:
# Training Results Analysis and Validation
import json
from pathlib import Path
import pandas as pd

print("📊 Analyzing training results...")

try:
    # Check for trained models
    models_dir = Path("models")
    if models_dir.exists():
        # Count models by type
        model_counts = {}
        total_models = 0
        
        for model_type in ['ppo', 'gru', 'lightgbm']:
            type_dir = models_dir / model_type
            if type_dir.exists():
                symbols = [d.name for d in type_dir.iterdir() if d.is_dir()]
                model_counts[model_type] = len(symbols)
                total_models += len(symbols)
                
                if symbols:
                    print(f"✅ {model_type.upper()}: {len(symbols)} models ({', '.join(symbols)})")
                else:
                    print(f"⚠️ {model_type.upper()}: No models found")
            else:
                print(f"❌ {model_type.upper()}: Directory not found")
                model_counts[model_type] = 0
        
        print(f"\n📈 Total models trained: {total_models}")
        
        # Check for training reports
        report_files = list(Path(".").glob("training_report_*.json"))
        if report_files:
            latest_report = max(report_files, key=lambda x: x.stat().st_mtime)
            print(f"\n📄 Latest training report: {latest_report.name}")
            
            try:
                with open(latest_report, 'r') as f:
                    report = json.load(f)
                
                summary = report.get('training_summary', {})
                if summary:
                    print(f"⏱️ Training time: {summary.get('total_training_time', 0):.1f}s")
                    print(f"📊 Avg validation score: {summary.get('average_validation_score', 0):.3f}")
                    print(f"🎯 Avg test score: {summary.get('average_test_score', 0):.3f}")
                
                model_performance = report.get('model_performance', {})
                if model_performance:
                    print("\n🏆 Model Performance Summary:")
                    for model_type, stats in model_performance.items():
                        print(f"  {model_type.upper()}: {stats.get('avg_validation_score', 0):.3f} avg score ({stats.get('count', 0)} models)")
                        
            except Exception as e:
                print(f"⚠️ Could not parse training report: {e}")
        else:
            print("⚠️ No training reports found")
            
        # Export validation
        if aws_available:
            print("\n☁️ Models ready for S3 deployment")
            print("🚀 Production servers can now import these models")
        else:
            print("\n💾 Models saved locally")
            print("💡 Configure AWS credentials to enable S3 export")
            
    else:
        print("❌ No models directory found - training may have failed")
        
except Exception as e:
    print(f"⚠️ Error analyzing results: {e}")

print("\n✅ Training analysis complete!")
print("\n🎯 Next steps:")
print("• Models are ready for production deployment")
print("• Check training reports for detailed performance metrics")
print("• Deploy to production trading servers via S3 or direct copy")